# PaMCP Section A — E0/E1/E2 Preflight

**Purpose:** run the read-only Section A inventory against the real GitHub repository, validate the preflight tests, and preserve a timestamped report. This notebook does not execute the legacy E0/E1/E2 scientific experiments and does not promote historical outputs to confirmatory evidence.

Run cells in order. The notebook checks out the research branch and writes a new report rather than overwriting an earlier one.

In [ ]:
from pathlib import Path
import subprocess, sys, json, datetime, os

REPO_URL = 'https://github.com/msellamiTN/taas-provenance-discovery.git'
BRANCH = 'research/pamcp-multibenchmark-v1'
ROOT = Path('/content/taas-provenance-discovery')

def run(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, check=True)

if ROOT.exists():
    run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT)
    run(['git', 'checkout', BRANCH], cwd=ROOT)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=ROOT)
else:
    run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(ROOT)])

commit = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
print('Repository:', ROOT)
print('Branch:', BRANCH)
print('Commit:', commit)


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], cwd=ROOT)
run([sys.executable, '-m', 'pytest', '-q', 'tests/test_section_a.py'], cwd=ROOT)


In [ ]:
campaign_dir = ROOT / 'results' / 'campaigns'
campaign_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
report_path = campaign_dir / f'section-a-preflight-{stamp}.json'
run([sys.executable, '-m', 'experiments.section_a', '--root', str(ROOT), '--output', str(report_path)], cwd=ROOT)
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps({
  'commit': report['repository']['git_commit'],
  'protocol': report['protocol'],
  'registry': report['registry'],
  'semantic_typing': report['semantic_typing'],
  'historical_result_count': len(report['historical_results']['files']),
  'safety': report['safety'],
  'report_path': str(report_path)
}, indent=2, ensure_ascii=False))


## Interpretation and acceptance gate

- `protocol.status == PASS` means the required protocol headings were found; it does not validate the scientific correctness of every protocol decision.
- `registry.status == PASS` means both expected registry source files are present. `PARTIAL` means at least one is missing.
- `semantic_typing.status == REVIEW_REQUIRED` means candidate annotation files exist, not that their independence or adjudication is proven.
- Historical E0/E1/E2 outputs remain `HISTORICAL_ONLY` until provenance compatibility is checked.

This notebook is a preflight only. It does not yet run E0, E1, or E2.